In [2]:
using SparseArrays, LinearAlgebra, LinearSolve
using OptimalTransport          # for Sinkhorn (if available)
using Enzyme                    # Reverse-mode on a scalar closure
using Random
using Distances                 # for pairwise squared distances
using LogExpFunctions           # for logsumexp in stable Sinkhorn
using Printf                    # for formatted printing

# ------------------ Grid & helpers ------------------
Nx, Ny, Nz = 20, 20, 20
x_range = range(-30, 30, length=Nx+1)
y_range = range(-30, 30, length=Ny+1)
z_range = range(0, 60, length=Nz+1)
xc = (x_range[1:end-1] .+ x_range[2:end]) ./ 2
yc = (y_range[1:end-1] .+ y_range[2:end]) ./ 2
zc = (z_range[1:end-1] .+ z_range[2:end]) ./ 2
idx(i,j,k) = (k-1)*(Nx*Ny) + (j-1)*Nx + i
N = Nx*Ny*Nz

# cell centers (N×3)
grid = let G = zeros(Float64, N, 3)
    for k in 1:Nz, j in 1:Ny, i in 1:Nx
        ii = idx(i,j,k)
        G[ii,1] = xc[i]; G[ii,2] = yc[j]; G[ii,3] = zc[k]
    end
    G
end

# Build the (dense) squared-Euclidean cost ONCE; guard against redefinition
if !@isdefined COST_W2
    const COST_W2 = pairwise(SqEuclidean(), grid', grid'; dims=2)  # N×N
end
@assert size(COST_W2,1) == N == size(COST_W2,2)
@assert maximum(COST_W2) > 0.0 "COST_W2 seems to be all zeros"

# ------------------ Lorenz velocity & params ------------------
Base.@kwdef mutable struct L63Params
    σ::Float64 = 10.0
    ρ::Float64 = 28.0
    β::Float64 = 8/3
end

@inline function lorenz_v_at_face(x::Float64,y::Float64,z::Float64,p::L63Params)
    vx = p.σ*(y - x)
    vy = x*(p.ρ - z) - y
    vz = x*y - p.β*z
    return vx, vy, vz
end

# ------------------ Sparse K(θ) with upwind split; column-sum zero ------------------
function build_K!(K::SparseMatrixCSC, p::L63Params)
    fill!(K.nzval, 0.0) # Zero out existing values
    dx = x_range[2]-x_range[1]; dy = y_range[2]-y_range[1]; dz = z_range[2]-z_range[1]

    # Loop over all *source* cells (s)
    for k_s in 1:Nz, j_s in 1:Ny, i_s in 1:Nx
        s = idx(i_s, j_s, k_s)
        x, y, z = grid[s, :] # Use grid centers directly
        diag_val = 0.0 # Accumulator for diagonal element K[s,s]

        # --- Flux calculation for each face based on upwinding ---

        # Face x+1/2 (between i_s and i_s+1)
        if i_s < Nx
            vx_face, _, _ = lorenz_v_at_face(x + dx / 2, y, z, p)
            flux_out = max(vx_face, 0.0) / dx # Flux out of s to s+dx
            flux_in = min(vx_face, 0.0) / dx  # Flux into s from s+dx
            K[idx(i_s + 1, j_s, k_s), s] += flux_out # Add flux TO neighbor
            diag_val -= flux_out # Subtract outgoing flux from diagonal
        else # Boundary condition (zero flux assumed implicitly by not adding)
             # vx_face, _, _ = lorenz_v_at_face(x + dx / 2, y, z, p)
             # flux_out = max(vx_face, 0.0) / dx
             # diag_val -= flux_out # Still need to account for potential outflow at boundary
        end

        # Face x-1/2 (between i_s-1 and i_s)
        if i_s > 1
            vx_face, _, _ = lorenz_v_at_face(x - dx / 2, y, z, p)
            flux_out = min(vx_face, 0.0) / dx # Flux out of s to s-dx (negative velocity)
            flux_in = max(vx_face, 0.0) / dx  # Flux into s from s-dx
            K[idx(i_s - 1, j_s, k_s), s] -= flux_out # Add flux TO neighbor (note minus sign)
            diag_val += flux_out # Add incoming flux (double negative = positive contribution)
        else # Boundary condition
             # vx_face, _, _ = lorenz_v_at_face(x - dx / 2, y, z, p)
             # flux_out = min(vx_face, 0.0) / dx
             # diag_val += flux_out
        end

        # Face y+1/2
        if j_s < Ny
            _, vy_face, _ = lorenz_v_at_face(x, y + dy / 2, z, p)
            flux_out = max(vy_face, 0.0) / dy
            flux_in = min(vy_face, 0.0) / dy
            K[idx(i_s, j_s + 1, k_s), s] += flux_out
            diag_val -= flux_out
        else
            # _, vy_face, _ = lorenz_v_at_face(x, y + dy / 2, z, p)
            # flux_out = max(vy_face, 0.0) / dy
            # diag_val -= flux_out
        end

        # Face y-1/2
        if j_s > 1
            _, vy_face, _ = lorenz_v_at_face(x, y - dy / 2, z, p)
            flux_out = min(vy_face, 0.0) / dy
            flux_in = max(vy_face, 0.0) / dy
            K[idx(i_s, j_s - 1, k_s), s] -= flux_out
            diag_val += flux_out
        else
            # _, vy_face, _ = lorenz_v_at_face(x, y - dy / 2, z, p)
            # flux_out = min(vy_face, 0.0) / dy
            # diag_val += flux_out
        end

        # Face z+1/2
        if k_s < Nz
            _, _, vz_face = lorenz_v_at_face(x, y, z + dz / 2, p)
            flux_out = max(vz_face, 0.0) / dz
            flux_in = min(vz_face, 0.0) / dz
            K[idx(i_s, j_s, k_s + 1), s] += flux_out
            diag_val -= flux_out
        else
            # _, _, vz_face = lorenz_v_at_face(x, y, z + dz / 2, p)
            # flux_out = max(vz_face, 0.0) / dz
            # diag_val -= flux_out
        end

        # Face z-1/2
        if k_s > 1
            _, _, vz_face = lorenz_v_at_face(x, y, z - dz / 2, p)
            flux_out = min(vz_face, 0.0) / dz
            flux_in = max(vz_face, 0.0) / dz
            K[idx(i_s, j_s, k_s - 1), s] -= flux_out
            diag_val += flux_out
        else
            # _, _, vz_face = lorenz_v_at_face(x, y, z - dz / 2, p)
            # flux_out = min(vz_face, 0.0) / dz
            # diag_val += flux_out
        end

        K[s, s] = diag_val # Set the diagonal: K_ss = - Σ_{i≠s} K_{is}
    end
    # Verify column sums are close to zero (debugging check)
    # max_col_sum_err = maximum(abs.(sum(K, dims=1)))
    # if max_col_sum_err > 1e-9
    #     @warn "Max column sum error in K is $max_col_sum_err"
    # end
    return K
end


# Build sparsity pattern ONCE
function prebuild_sparsity_pattern()
    rows = Int[]; cols = Int[];
    push_triplet(r,c) = (push!(rows,r); push!(cols,c))
    for k in 1:Nz, j in 1:Ny, i in 1:Nx
        s = idx(i,j,k)
        push_triplet(s,s) # Diagonal
        # Neighbors
        if i<Nx; push_triplet(idx(i+1,j,k), s); end
        if i>1 ; push_triplet(idx(i-1,j,k), s); end
        if j<Ny; push_triplet(idx(i,j+1,k), s); end
        if j>1 ; push_triplet(idx(i,j-1,k), s); end
        if k<Nz; push_triplet(idx(i,j,k+1), s); end
        if k>1 ; push_triplet(idx(i,j,k-1), s); end
    end
    # Create sparse matrix with zero values just to establish pattern
    # Ensure dimensions match N x N
    K_pattern = sparse(rows, cols, 0.0, N, N)
    return K_pattern
end

# Store the pattern globally
if !@isdefined K_SPARSITY_PATTERN
    const K_SPARSITY_PATTERN = prebuild_sparsity_pattern()
end

function build_M(p::L63Params; c::Float64=NaN)
    # Use the precomputed pattern
    K = copy(K_SPARSITY_PATTERN) # IMPORTANT: copy the pattern
    build_K!(K, p) # Fill values into K

    # CFL scaling based on maximum absolute diagonal value of K
    diag_K = diag(K)
    # Filter out potential zeros if system stops in some cells
    abs_diag_K_nz = abs.(diag_K[diag_K .!= 0.0])
    vmax = isempty(abs_diag_K_nz) ? 0.0 : maximum(abs_diag_K_nz)

    # If vmax is zero or very small, use a default c or handle appropriately
    if vmax < 1e-12
        # @warn "Max diagonal value of K is near zero. Using default scaling."
        c_default = 1.0 # Or some other reasonable default
        cval = isfinite(c) ? c : c_default
    else
        cval = isfinite(c) ? c : 1.0 / (vmax + 1e-9) # Add epsilon for stability
    end

    M = I + cval * K
    # Ensure M is CSC format which KLUFactorization prefers
    return SparseMatrixCSC(M), cval
end


# ------------------ Stationary solve & adjoint ------------------
struct PFForward{T}
    A::SparseMatrixCSC{T,Int} # Store the matrix A = (1-ε)M - I used in the solve
    ρ::Vector{T}              # Store the resulting normalized density
end

function forward_stationary(M::SparseMatrixCSC{T,Int}, ε::T) where {T}
    n = size(M,1)
    A = (1-ε)*M - I
    b = -(ε/n)*ones(T,n)

    # Robustness: Check if A is singular or ill-conditioned before solve? Maybe not needed due to ε.
    local ρu
    try
        # Use KLUFactorization which is good for sparse systems
        prob = LinearProblem(A, b)
        sol = solve(prob, KLUFactorization())
        ρu = sol.u
    catch e
        @error "Linear solve failed in forward_stationary: $e"
        # Potentially return an error indicator or a default value?
        # For now, rethrow to see the error cause.
        rethrow(e)
    end

    # Ensure positivity and normalize
    ρ_clipped = max.(ρu, 0.0) # Clip potential small negatives from numerical noise
    s = sum(ρ_clipped)
    if s < eps(T) # Avoid division by zero/NaN if solution is near zero everywhere
        @warn "Sum of solved ρ_clipped is near zero ($s). Returning uniform distribution."
        # Fallback to uniform distribution
        ρ = fill(one(T)/n, n)
    else
        ρ = ρ_clipped ./ s
    end

    return PFForward(A, ρ) # Store A and the final normalized ρ
end


# --- Robust OT wrapper: try OptimalTransport; fallback LOG-STABILIZED dense Sinkhorn ---
function ot_phi(ρ::AbstractVector{Float64}, ρstar::AbstractVector{Float64};
                ε_ent::Float64=1e-2, maxiter::Int=2000, tol::Float64=1e-9)

    # --- Input checks ---
    tiny_err = 1e-7 # Allow for small numerical deviations from 1.0
    sum_ρ = sum(ρ)
    sum_ρstar = sum(ρstar)
    if !(abs(sum_ρ - 1.0) < tiny_err) @warn "Input ρ not normalized (sum=$sum_ρ)" end
    if !(abs(sum_ρstar - 1.0) < tiny_err) @warn "Input ρstar not normalized (sum=$sum_ρstar)" end
    @assert length(ρ) == length(ρstar) == size(COST_W2,1) "Dimension mismatch"
    # Allow small negative values due to numerical noise, clip them
    if any(ρ .< -tiny_err) @warn "Input ρ has significant negative values" end
    if any(ρstar .< -tiny_err) @warn "Input ρstar has significant negative values" end

    # Ensure inputs are non-negative and normalized before passing to solvers
    ρ_clean = max.(ρ, 0.0); ρ_clean ./= sum(ρ_clean)
    ρstar_clean = max.(ρstar, 0.0); ρstar_clean ./= sum(ρstar_clean)


    # 1) Try library Sinkhorn
    try
        # Explicitly choose SinkhornGibbs
        alg = OptimalTransport.SinkhornGibbs()

        # Ensure inputs are Float64
        ρ_f64 = convert(Vector{Float64}, ρ_clean)
        ρstar_f64 = convert(Vector{Float64}, ρstar_clean)
        C_f64 = convert(Matrix{Float64}, COST_W2)
        ε_ent_f64 = convert(Float64, ε_ent)


        res = OptimalTransport.sinkhorn(ρ_f64, ρstar_f64, C_f64, ε_ent_f64;
                                        alg=alg,
                                        maxiter=maxiter,
                                        rtol=tol, # Use rtol for relative tolerance
                                        check_convergence=50, # Check less often
                                        assume_convergence=true) # Assume converges unless error throws

        # Extract potential φ = ε * log(u). Add eps() for numerical stability.
        # Ensure res.u does not contain zeros or negatives before log
        u_safe = max.(res.u, eps(Float64))
        φ_raw = ε_ent_f64 .* log.(u_safe)
        # Normalize potential φ by subtracting its weighted mean wrt ρ
        φ = φ_raw .- dot(φ_raw, ρ_f64) # Use dot product for weighted mean

        L = res.reg_ot_cost

        # Robustness check
        if !isfinite(L) || any(!isfinite, φ)
             @warn "OptimalTransport.sinkhorn returned non-finite values (L=$L). Using fallback."
             error("Fallback needed") # Force fallback
        end

        # println("Used OptimalTransport.sinkhorn successfully.") # Debug print
        return L, φ
    catch e
        # 2) Fallback: LOG-STABILIZED dense Sinkhorn (More robust but potentially slow)
        @warn "OptimalTransport.sinkhorn failed ($(typeof(e))). Using fallback LOG-STABILIZED Sinkhorn."
        # showerror(stdout, e, catch_backtrace()) # Uncomment for detailed error

        Nloc = length(ρ_clean)
        f = zeros(Float64, Nloc)  # Potential φ = f (associated with ρ)
        g = zeros(Float64, Nloc)  # Potential ψ = g (associated with ρstar)
        K = -COST_W2 ./ ε_ent     # Cost scaled by -1/ε for log-space Gibbs kernel

        # Precompute logs - essential for stability
        # Add small epsilon before log to handle exact zeros
        log_ρ_clean = log.(max.(ρ_clean, eps(Float64)))
        log_ρstar_clean = log.(max.(ρstar_clean, eps(Float64)))

        # Log-Sinkhorn iterations (stabilized updates)
        for it in 1:maxiter
            g_old = copy(g) # For convergence check

            # Update g = -ε * logsumexp( (f_i + K_{ij}) / ε ) + ε*log(ρstar_j)
            f_over_eps = f ./ ε_ent
            logsumexp_f = logsumexp(K .+ f_over_eps', dims=1)[:]
            g .= .-ε_ent .* logsumexp_f .+ ε_ent .* log_ρstar_clean

            # Update f = -ε * logsumexp( (g_j + K_{ij}) / ε ) + ε*log(ρ_i)
            g_over_eps = g ./ ε_ent
            logsumexp_g = logsumexp(K .+ g_over_eps', dims=2)[:]
            f .= .-ε_ent .* logsumexp_g .+ ε_ent .* log_ρ_clean

             # Convergence check
             if it % 50 == 0
                 delta_g = norm(g - g_old, Inf)
                 if delta_g < tol
                     #println("Fallback Sinkhorn converged at iter $it")
                     break
                 end
             end
             if it == maxiter
                 final_delta_g = norm(g - g_old, Inf)
                 @warn "Fallback Sinkhorn reached maxiter ($maxiter) without converging to tol=$tol (final delta=$final_delta_g)"
             end
        end

        # Normalize potential φ = f by subtracting weighted mean wrt ρ
        φ = f .- dot(f, ρ_clean) # Use dot product

        # Calculate loss from potentials: L = φ⋅ρ + g⋅ρstar (Regularized OT cost)
        L = dot(f, ρ_clean) + dot(g, ρstar_clean) # Use cleaned inputs

        if !isfinite(L) || any(!isfinite, φ)
            @error "Fallback Sinkhorn failed to produce finite results (L=$L). Check inputs/parameters (ε_ent=$ε_ent)."
             return 0.0, zeros(Nloc) # Return zero gradient/loss?
        end
        return L, φ
    end
end

function adjoint_lambda(forw::PFForward{T}, φ::AbstractVector{T}) where {T}
    ρ = forw.ρ
    # Project φ to be orthogonal to ρ (ensures consistency for the adjoint solve)
    φbar = φ .- dot(φ, ρ) # Use dot product correctly
    AT = sparse(transpose(forw.A)) # Ensure CSC format for KLU

    # Check if φbar is sufficiently orthogonal to the null space (ρ) of A^T - I?
    # Not strictly necessary if φ comes from OT dual, should be close.

    local λ
    try
        prob_adj = LinearProblem(AT, φbar)
        sol_adj = solve(prob_adj, KLUFactorization())
        λ = sol_adj.u
        # Optional: Project λ to be orthogonal to 1 (null space of A^T-I is span{1})
        # This selects a unique solution from the affine subspace.
        # λ .-= sum(λ) / length(λ)
    catch e
         @error "Linear solve failed in adjoint_lambda: $e"
         # Handle error: maybe return zero vector or rethrow
         rethrow(e)
    end
    return λ
end


# ------------------ Enzyme-powered parameter gradient ------------------
# ∇θ L = (1-ε) * ∇θ g(θ) with g(θ) = λᵀ M(θ) ρ

function g_scalar(p::L63Params, ρ::Vector{Float64}, λ::Vector{Float64}; c::Float64)
    # Rebuild M with the *current* parameters p and the *same* scaling c used in forward pass
    M, _ = build_M(p; c=c)
    # Compute the scalar value λᵀ * M * ρ
    return dot(λ, M * ρ)
end


# ------------------ Simple GD loop ------------------
# --- Coordinate Gradient Descent loop (Paper Sec 6.3.2) ---
function fit!(p::L63Params, ρstar::Vector{Float64};
              epochs::Int=30,                     # Number of full cycles through params
              step_sizes = (σ=0.5, ρ=5.0, β=0.05), # Tuned step sizes (NamedTuple)
              ε::Float64=1e-6,                    # Teleportation param
              ε_ent::Float64=1e-2)                # Sinkhorn regularization

    # Allocate gradient buffer once
    dp = L63Params(0.0, 0.0, 0.0)
    # Get initial loss
    # Ensure dp buffer is passed here too
    L_initial, _, _ = loss_and_grad_enzyme(p, ρstar; ε=ε, ε_ent=ε_ent, dp=dp)
    @info "Starting Coordinate GD. Initial Params: σ=$(@sprintf("%.4f", p.σ)), ρ=$(@sprintf("%.4f", p.ρ)), β=$(@sprintf("%.4f", p.β)), Initial Loss: $(@sprintf("%.6e", L_initial))"

    history = [] # Store (epoch, loss, params, grad_norm)

    for epoch in 1:epochs
        local L_iter = NaN # Store loss from the last update in the epoch

        # --- Update σ ---
        try
            _, (gσ, _, _), _ = loss_and_grad_enzyme(p, ρstar; ε=ε, ε_ent=ε_ent, dp=dp)
            p.σ -= step_sizes.σ * gσ
        catch e
             @error "Error during σ update at epoch $epoch: $e"
             break # Stop optimization if gradient fails
        end

        # --- Update ρ ---
         try
            _, (_, gρ, _), _ = loss_and_grad_enzyme(p, ρstar; ε=ε, ε_ent=ε_ent, dp=dp)
            p.ρ -= step_sizes.ρ * gρ
        catch e
             @error "Error during ρ update at epoch $epoch: $e"
             break
        end

        # --- Update β ---
         try
            L_iter, (_, _, gβ), _ = loss_and_grad_enzyme(p, ρstar; ε=ε, ε_ent=ε_ent, dp=dp)
            p.β -= step_sizes.β * gβ
        catch e
             @error "Error during β update at epoch $epoch: $e"
             break
        end

        # --- Logging after full epoch ---
        # Recompute full gradient for accurate norm reporting
        _, (gσ_final, gρ_final, gβ_final), _ = loss_and_grad_enzyme(p, ρstar; ε=ε, ε_ent=ε_ent, dp=dp)
        grad_norm = sqrt(gσ_final^2 + gρ_final^2 + gβ_final^2)

        # Store history
        push!(history, (epoch=epoch, loss=L_iter, params=deepcopy(p), grad_norm=grad_norm))

        # Check for NaN/Inf parameters or loss
        if any(!isfinite, (p.σ, p.ρ, p.β)) || !isfinite(L_iter)
            @error "Non-finite values encountered at epoch $epoch. Stopping. Params: σ=$(p.σ), ρ=$(p.ρ), β=$(p.β), Loss: $L_iter"
            break
        end

        # Log progress periodically
        if epoch % 1 == 0 || epoch == epochs # Log every epoch
             @info "Epoch $(@sprintf("%3d", epoch))  L=$(@sprintf("%.6e", L_iter))  θ=(σ=$(@sprintf("%.4f", p.σ)), ρ=$(@sprintf("%.4f", p.ρ)), β=$(@sprintf("%.4f", p.β)))  ‖g‖=$(@sprintf("%.4e", grad_norm))"
        end
    end
    println("Coordinate GD finished.")
    return p, history
end

# Helper function needs to accept dp buffer AND ZERO IT OUT internally
function loss_and_grad_enzyme(p::L63Params, ρstar::Vector{Float64};
                              ε::Float64=1e-6, ε_ent::Float64=1e-2,
                              dp::L63Params = L63Params(0.0,0.0,0.0)) # Provide buffer

    local M, c, forw, ρ, loss, φ, λ
    try
        M, c = build_M(p)
    catch e
        @error "Error in build_M with params σ=$(p.σ), ρ=$(p.ρ), β=$(p.β): $e"
        rethrow(e)
    end
    try
        forw = forward_stationary(M, ε)
        ρ = forw.ρ
        # safety normalize just in case
        sρ = sum(ρ);
        if sρ < eps(Float64)
             @warn "Forward solve resulted in near-zero ρ sum ($sρ). Setting uniform."
             ρ = fill(1.0/N, N)
        elseif abs(sρ - 1.0) > 1e-8
             #@warn "Renormalizing ρ in loss_and_grad (sum was $sρ)"
             ρ ./= sρ
        end
        # Ensure ρ is strictly positive for log in Sinkhorn/potential extraction
        ρ = max.(ρ, eps(Float64)); ρ ./= sum(ρ) # Ensure positivity and re-normalize

    catch e
         @error "Error in forward_stationary: $e"
         rethrow(e)
    end

    try
        loss, φ = ot_phi(ρ, ρstar; ε_ent=ε_ent)
    catch e
         @error "Error in ot_phi: $e"
         rethrow(e)
    end
    try
         λ = adjoint_lambda(forw, φ)
    catch e
         @error "Error in adjoint_lambda: $e"
         rethrow(e)
    end


    # scalar closure; Enzyme captures ρ, λ, c
    gs(p_) = g_scalar(p_, ρ, λ; c=c)

    # Zero out buffer before AD call *inside* the function
    dp.σ=0.0; dp.ρ=0.0; dp.β=0.0

    # AD Call
    try
        # Using Duplicated as it worked before
        Enzyme.autodiff(Reverse, gs, Active, Duplicated(p, dp))
    catch e
        @error "Error during Enzyme.autodiff: $e"
        # Potentially return zero gradient if AD fails?
        dp.σ=0.0; dp.ρ=0.0; dp.β=0.0
        # rethrow(e) # Rethrow to halt execution
    end


    # chain rule factor (1-ε)
    # Check if dp contains NaNs before scaling
    if any(!isfinite, (dp.σ, dp.ρ, dp.β))
        @warn "NaN detected in dp before scaling at params σ=$(p.σ), ρ=$(p.ρ), β=$(p.β)"
        # Handle NaN gradient, e.g., return zero gradient?
        gσ=0.0; gρ=0.0; gβ=0.0
    else
        gσ = (1-ε) * dp.σ
        gρ = (1-ε) * dp.ρ
        gβ = (1-ε) * dp.β
    end

    # Final check for NaN loss
    if !isfinite(loss)
        @warn "NaN loss detected at params σ=$(p.σ), ρ=$(p.ρ), β=$(p.β). Returning 0 gradient."
        gσ=0.0; gρ=0.0; gβ=0.0
        loss = 0.0 # Or some large penalty value?
    end


    return loss, (gσ, gρ, gβ), ρ
end


# ------------------ Demo target (inverse crime ok for testing) ------------------
p_true = L63Params()
println("Building M_true...")
M_true, c_true = build_M(p_true) # Capture c_true if needed later, though maybe not
println("Solving for rho_star...")
forw_true = forward_stationary(M_true, 1e-6)
ρstar = forw_true.ρ
# Ensure ρstar is normalized (forward_stationary should handle this, but double check)
ρstar ./= sum(ρstar)
println("rho_star computed. Sum = $(sum(ρstar)), Min = $(minimum(ρstar)), Max = $(maximum(ρstar))")


# initialize and run
println("Initializing parameters...")
p0 = L63Params(σ=5.0, ρ=20.0, β=1.0) # Start further away

println("Starting optimization...")
# Use the corrected keyword arguments: epochs, step_sizes
p_est, history = fit!(p0, ρstar;
                      epochs=30,
                      step_sizes=(σ=0.5, ρ=5.0, β=0.05), # Corrected args
                      ε=1e-6,
                      ε_ent=1e-2)

println("\nOptimization finished.")
println("True parameters: σ=$(p_true.σ), ρ=$(p_true.ρ), β=$(@sprintf("%.4f", p_true.β))")
println("Estimated parameters: σ=$(@sprintf("%.4f", p_est.σ)), ρ=$(@sprintf("%.4f", p_est.ρ)), β=$(@sprintf("%.4f", p_est.β))")


Building M_true...
Solving for rho_star...
rho_star computed. Sum = 1.0, Min = 2.400815644668214e-10, Max = 0.009905981532995777
Initializing parameters...
Starting optimization...


┌ Warning: OptimalTransport.sinkhorn failed (MethodError). Using fallback LOG-STABILIZED Sinkhorn.
└ @ Main /Users/niklasviebig/master_thesis/LorenzParameterEstimation/examples_climate/pdf/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W0sZmlsZQ==.jl:302
┌ Error: Error in ot_phi: InterruptException()
└ @ Main /Users/niklasviebig/master_thesis/LorenzParameterEstimation/examples_climate/pdf/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W0sZmlsZQ==.jl:499


InterruptException: InterruptException:

In [ ]:
using Plots

epochs = [h.epoch for h in history]
losses = [h.loss for h in history]
params_hist = [h.params for h in history]
σ_hist = [p.σ for p in params_hist]
ρ_hist = [p.ρ for p in params_hist]
β_hist = [p.β for p in params_hist]

p1 = plot(epochs, losses, yscale=:log10, title="Loss", xlabel="Epoch", label="W2 Regularized")
p2 = plot(epochs, σ_hist, label="σ", title="Parameters", xlabel="Epoch")
plot!(p2, epochs, ρ_hist, label="ρ")
plot!(p2, epochs, β_hist, label="β")
hline!(p2, [p_true.σ, p_true.ρ, p_true.β], linestyle=:dash, label=["σ true" "ρ true" "β true"])
plot(p1, p2, layout=(2,1))